#MSSA data from HCAI API service

Gather the entire MSSA dataset

In [0]:
import requests
from pyspark.sql import SparkSession
import pandas as pd

spark = SparkSession.builder.getOrCreate()

mssa_url = "https://services5.arcgis.com/fMBfBrOnc6OOzh7V/arcgis/rest/services/Medical_Service_Study_Areas_/FeatureServer/0/query"

total_record = 9129
offset = 0 
record_holder = []
attempt = 0

while offset < total_record:
    mssa_response = requests.get(mssa_url)
    params = {
        "where": "1=1",
        "outFields": "*",
        "outSR": "4326",
        "f": "json",
        "resultOffset" : offset,
        "resultRecordCount": 2000
        }
    mssa_response = requests.get(mssa_url, params=params)

    print(f"Attempt {attempt} Status code: {mssa_response.status_code}")
    mssa_data = mssa_response.json()
    record_holder.extend(mssa_data['features'])
    offset = len(record_holder)
    attempt += 1

print(f"Total records: {len(record_holder)}")


Convert the data dictionaries into dataframe

In [0]:

mssa_features_df = pd.json_normalize(record_holder)

mssa_features_df.columns

Rename the mssa dataframe column labels. 

In [0]:
col_rename_dict = {
    'attributes.FID': 'FID',
    'attributes.STATEFP': 'STATEFP',
    'attributes.COUNTYFP': 'COUNTYFP',
    'attributes.COUNTYNM': 'COUNTYNM',
    'attributes.TRACTCE': 'TRACTCE',
    'attributes.GEOID': 'GEOID',
    'attributes.ALAND': 'ALAND',
    'attributes.AWATER': 'AWATER', 
    'attributes.ASQMI': 'ASQMI',
    'attributes.INTPTLAT': 'INTPTLAT',
    'attributes.INTPTLON': 'INTPTLON',
    'attributes.MSSAID': 'MSSAID',
    'attributes.MSSANM': 'MSSANM',
    'attributes.DEFINITION': 'DEFINITION',
    'attributes.TOTALPOVPO': 'TOTALPOVPO',
    'attributes.Shape__Area': 'Shape_Area', 
    'attributes.Shape__Length': 'Shape_Length',
    'geometry.rings': 'geometry_rings'}


In [0]:
mssa_features_df.rename(columns=col_rename_dict, inplace=True)
mssa_features_df.head()


In [0]:
mssa_features_spark_df = spark.createDataFrame(mssa_features_df)
mssa_features_spark_df.printSchema()

In [0]:
mssa_features_spark_df.write.mode("overwrite").saveAsTable("ca_healthcare_fac_bronze.mssa_data_bronze.mssa_geo")

To download the data for handoff, save them to the volume

In [0]:
mssa_geo = spark.read.table("ca_healthcare_fac_bronze.mssa_data_bronze.mssa_geo").toPandas()
mssa_geo.shape


In [0]:
volume_path = "/Volumes/ca_healthcare_fac_silver/default/silver_export"

mssa_geo.to_csv(f"{volume_path}/mssa_geo.csv", index=False)
